In [ ]:
# 1. Libraries
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# 2. Load dataset
df = pd.read_csv("match_prediction_features_X.csv")
print(df.shape)
display(df.head())

# 3. Find text and numeric columns
text_cols = df.select_dtypes(include="object").columns
num_cols = df.select_dtypes(include=np.number).columns

print("Text:", list(text_cols))
print("Numeric:", list(num_cols))

# 4. NLP: combine text columns
df["text"] = df[text_cols].fillna("").astype(str).agg(" ".join, axis=1)
df["text"] = df["text"].apply(
    lambda x: re.sub(r"[^a-zA-Z\s]", " ", x.lower())
)

# 5. TF-IDF
tfidf = TfidfVectorizer(max_features=500, stop_words="english")
X_text = tfidf.fit_transform(df["text"]).toarray()

# 6. Numeric features
X_num = df[num_cols].fillna(0)
X_num = StandardScaler().fit_transform(X_num)

# 7. Combine NLP + numeric features
X = np.hstack((X_num, X_text))

# 8. K-Means
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X)

# 9. Show clusters
print(df["cluster"].value_counts())
display(df[["cluster"]].head())

# 10. Visualise clusters
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(8,5))
plt.scatter(X_pca[:,0], X_pca[:,1], c=df["cluster"])
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("K-Means Match Clusters")
plt.show()

# 11. Save result
df.to_csv("match_prediction_features_with_clusters.csv", index=False)
print("Saved successfully!")